# Train the CTC phoneme recogniser (Iqra'Eval 2025 data only)

**Settings:** Accelerator = *GPU T4 x2*, Internet = *On*.
**Input:** attach the `iqraeval-data` notebook output (Add Input → Your Work → Notebooks).
**Secrets:** `HF_TOKEN` (write access), `WANDB_API_KEY` (optional).

Run with **Save & Run All (Commit)**. Checkpoints go to `HUB_REPO`; running again resumes from it.
Use a new `RUN_NAME` for each new experiment (it also names the private Hub repo).

In [ ]:
RUN_NAME = "ctc-xlsr"        # also names the Hub repo: <your HF username>/iqraeval-<RUN_NAME>
HUB_REPO = None              # or set "user/repo" explicitly; None = derived from the HF token owner
WANDB_PROJECT = "iqraeval-mdd"
REPO_URL = "https://github.com/hBouanane/wav2vec2-mispronunciation-detector"   # your fork, if any
BRANCH = "main"
EXTRA = "--encoder elgeish/wav2vec2-large-xlsr-53-arabic --max-steps 20000"

# Other configurations used in the experiments (see README):
# RUN_NAME = "ctc-w2vbert-ttsorig"
# EXTRA = "--encoder facebook/w2v-bert-2.0 --tts-labels original --freeze-layers 6 --batch-size 4 --grad-accum 4 --warmup 500 --max-steps 12000"
# RUN_NAME = "ctc-muaalem-qps"
# EXTRA = "--encoder obadx/muaalem-model-v3_2 --target qps --freeze-layers 12 --batch-size 4 --grad-accum 4 --max-steps 20000"


In [ ]:
import os
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
try:
    os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
    WANDB_ARGS = f"--wandb-project {WANDB_PROJECT} --wandb-name {RUN_NAME}"
except Exception:
    print("no WANDB_API_KEY secret - training without wandb"); WANDB_ARGS = ""
!rm -rf repo && git clone -q -b $BRANCH $REPO_URL repo
!git -C repo log --oneline -1
!git clone -q --depth 1 https://github.com/Iqra-Eval/interspeech_IqraEval
!pip install -q jiwer wandb
!find /kaggle/input -maxdepth 6 -type d -name data | head
!nvidia-smi --query-gpu=name,memory.total --format=csv; df -h /kaggle/working | tail -1
from huggingface_hub import whoami
HUB_REPO = HUB_REPO or f"{whoami()['name']}/iqraeval-{RUN_NAME}"
print("checkpoints go to", HUB_REPO)

In [ ]:
RUN = "/kaggle/working/run"
!cd repo && PYTHONUNBUFFERED=1 accelerate launch --multi_gpu --num_processes 2 --mixed_precision fp16 train.py     --out $RUN --mdd-eval ../interspeech_IqraEval/mdd_eval --hub-repo $HUB_REPO $WANDB_ARGS $EXTRA     2>&1 | grep --line-buffered -v -i warn

In [ ]:
!cat $RUN/best.json 2>/dev/null | head -20; tail -3 $RUN/eval_log.jsonl 2>/dev/null